# Project 9 — Amazon ML Challenge 2024 (Entity Extraction from Images)

**From the AI/ML Projects Ebook by Sreemanti.**

> Extract attributes (e.g. weight, voltage, dimensions) directly from product images. RAG didn't cut it — fine-tuning did.

---

**Difficulty:** 9/10 · **Resume value:** 8/10 · **Learning value:** 10/10 · **Impact:** 9/10


## What you'll learn
- Vision–language pipelines
- When RAG is the wrong tool and fine-tuning wins
- LoRA/PEFT-style fine-tuning of multimodal models
- Building structured-output extractors

## Tech stack
`transformers`, `peft`, `accelerate`, `bitsandbytes`, `Pillow`, `pytesseract` (baseline).


## 1. Setup


In [ ]:
# !pip install transformers peft accelerate bitsandbytes pillow pytesseract -q
import os, json, re
from PIL import Image
import pytesseract


## 2. OCR baseline
Before any fine-tuning, see how far OCR + regex gets you. This is the honest baseline you'll beat.


In [ ]:
def ocr_baseline(image_path: str, attribute: str):
    text = pytesseract.image_to_string(Image.open(image_path))
    patterns = {
        'item_weight':    r'(\d+(?:\.\d+)?)\s*(g|kg|lb|oz)',
        'voltage':        r'(\d+(?:\.\d+)?)\s*(V|kV)',
        'wattage':        r'(\d+(?:\.\d+)?)\s*(W|kW)',
        'item_volume':    r'(\d+(?:\.\d+)?)\s*(ml|l|L)',
    }
    p = patterns.get(attribute)
    if not p: return None
    m = re.search(p, text, re.I)
    return f'{m.group(1)} {m.group(2)}' if m else None


## 3. Vision-language model with structured output
Replace the OCR baseline with a small vision-language model (e.g. Qwen2-VL-2B, Llava, Phi-3-Vision). Below is the inference shape — fine-tune on the challenge's labelled set.


In [ ]:
# Pseudocode-style scaffold. Set your own model + adapter paths.
from transformers import AutoProcessor, AutoModelForVision2Seq
import torch

MODEL = 'Qwen/Qwen2-VL-2B-Instruct'  # or any open VLM you can run
processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForVision2Seq.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='auto')

PROMPT = (
    'Look at the product image and extract the requested attribute as JSON.\n'
    'Schema: {"value": number, "unit": string} or {"value": null} if not present.\n'
    'Attribute: {attribute}'
)

def predict(image_path, attribute):
    img = Image.open(image_path).convert('RGB')
    msgs = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': PROMPT.format(attribute=attribute)}]}]
    inputs = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_tensors='pt', images=[img]).to(model.device)
    out = model.generate(**inputs, max_new_tokens=64)
    return processor.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


## 4. Fine-tuning sketch (LoRA)


In [ ]:
# Use the official challenge dataset. Below is the LoRA setup outline.
from peft import LoraConfig, get_peft_model
lora = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj', 'v_proj'], lora_dropout=0.05, bias='none')
# model = get_peft_model(model, lora)
# Then a standard HF Trainer / SFTTrainer loop on (image, prompt, target_json) tuples.


## 5. Stretch goals
- Compare OCR-only vs VLM-zero-shot vs VLM-fine-tuned on a fixed eval split
- Add post-processing for unit normalisation
- Submit to Amazon ML Challenge 2025
